# CardioLens — Exploratory Data Analysis

Quick EDA on the UCI Cleveland Heart Disease dataset before we commit to the modeling pipeline.

What we're checking:
1. Class balance
2. Missing-value patterns
3. Feature distributions, split by target
4. Correlation structure
5. Sex / age subgroup composition (sanity check before fairness audit)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_raw, clean

df = clean(load_raw())
df.shape, df.columns.tolist()

In [ ]:
# Class balance
df['target'].value_counts(normalize=True).round(3)

In [ ]:
# Distributions split by target
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']):
    sns.kdeplot(data=df, x=col, hue='target', ax=ax, fill=True, alpha=0.4)
    ax.set_title(col)
fig.tight_layout()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature correlation')
plt.show()

In [ ]:
# Subgroup composition — important context for the fairness audit
import pandas as pd

df['age_bucket'] = pd.cut(df['age'], bins=[0, 45, 55, 65, 100], labels=['<45', '45-54', '55-64', '65+'])
df.groupby(['sex', 'age_bucket'], observed=True)['target'].agg(['count', 'mean']).round(3)